# Big Five personality prediction from Reddit text

## Experimental notebook for the revised paper

This notebook records the analyses used in *A Comparative Evaluation of Traditional Machine Learning Models for Big Five Personality Prediction from Reddit Text*. It uses the original PANDORA author profiles and Reddit comments, not the earlier Kaggle-based exploratory sample.

The code cells are preserved from the working Colab notebook, in their original order. The short notes identify what each cell does and why it appears in the paper. Some follow-up cells were run after a Colab restart and use artifacts saved by earlier cells. This is a record of the actual analysis, not a claim that every cell has been verified in a fresh runtime.

**Data access.** The notebook expects `author_profiles.csv` and `pandora_filtered_comments.csv` in `/content/drive/MyDrive/PANDORA_Original/`. Those files are not included here. Obtain access from the PANDORA authors and follow the dataset's usage conditions; do not upload author-level source data or trained artifacts publicly without checking those conditions.

**Evaluation.** The main experiment uses 1,094 training authors and 274 held-out test authors. Hyperparameters are selected on training data; the test partition is reserved for reporting results. Later sections document additional analyses requested during revision.


### Connect Google Drive

Mount the Drive folder used for the original PANDORA files and experiment outputs.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


### Select profiles with complete Big Five labels

Read the author profile file and identify users with all five personality scores available.


In [ ]:

import pandas as pd

profiles_path = "/content/drive/MyDrive/PANDORA_Original/author_profiles.csv"

profiles = pd.read_csv(profiles_path, low_memory=False)

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

print("Ukupan broj profila:", len(profiles))
print("Broj jedinstvenih autora:", profiles["author"].nunique())

complete = profiles.dropna(subset=["author"] + traits)
percentile_profiles = complete[
    complete["test_scale"].astype(str).str.strip().eq("percentiles")
]

print("Profili sa svih 5 oznaka:", len(complete))
print("Kompletni profili sa percentile skalom:", len(percentile_profiles))
print("Jedinstveni autori u odabranim profilima:",
      percentile_profiles["author"].nunique())

print("\nRasponi Big Five oznaka:")
print(percentile_profiles[traits].agg(["min", "max"]))


### Check the reported label scales

Inspect `test_scale` before filtering. This check helps distinguish percentile scores from other test formats.


In [ ]:

print("Vrijednosti test_scale za kompletne profile:")
print(complete["test_scale"].value_counts(dropna=False))

print("\nVrijednosti test_scale u cijelom datasetu:")
print(profiles["test_scale"].value_counts(dropna=False))


### Normalize personality targets

Keep the percentile-based profiles and divide their five scores by 100.


In [ ]:

# Izdvajamo autore s kompletnim Big Five oznakama
selected_profiles = percentile_profiles[
    ["author"] + traits
].copy()

# Pretvaramo Big Five vrijednosti sa skale 0–100 na 0–1
selected_profiles[traits] = selected_profiles[traits] / 100.0

print("Broj odabranih autora:", len(selected_profiles))
print("Broj jedinstvenih autora:",
      selected_profiles["author"].nunique())

print("\nRasponi normaliziranih oznaka:")
print(selected_profiles[traits].agg(["min", "max"]))


### Load the Reddit comments

Read the author and comment-body fields from `pandora_filtered_comments.csv`.


In [ ]:

import pandas as pd

comments_path = "/content/drive/MyDrive/PANDORA_Original/pandora_filtered_comments.csv"

# Učitavamo samo kolone potrebne za istraživanje
comments = pd.read_csv(
    comments_path,
    usecols=["author", "body", "subreddit", "id"],
    low_memory=False
)

print("Ukupan broj komentara:", len(comments))
print("Broj jedinstvenih autora:", comments["author"].nunique())

print("\nNedostajuće vrijednosti:")
print(comments.isna().sum())


### Clean and match comments to profiles

Discard empty comment bodies and match the remaining text to authors with usable labels.


In [ ]:

# Uklanjamo komentare bez teksta
clean_comments = comments.dropna(subset=["body"]).copy()

# Uklanjamo komentare koji su prazni ili sadrže samo razmake
clean_comments = clean_comments[
    clean_comments["body"].str.strip().ne("")
].copy()

# Povezujemo komentare i Big Five oznake preko autora
merged_df = clean_comments.merge(
    selected_profiles,
    on="author",
    how="inner",
    validate="many_to_one"
)

print("Broj komentara nakon povezivanja:", len(merged_df))
print("Broj jedinstvenih autora:", merged_df["author"].nunique())

print("\nBroj komentara po autoru:")
print(merged_df.groupby("author").size().describe())


### Inspect author activity thresholds

Count how many authors remain at each minimum-comment threshold.


In [ ]:

comments_per_author = merged_df.groupby("author").size()

for minimum in [1, 10, 20, 50, 100, 200]:
    count = (comments_per_author >= minimum).sum()
    print(f"Autori sa najmanje {minimum} komentara: {count}")


### Build one document per author

Keep authors with at least 20 comments, sample up to 100 comments per author, and aggregate their text.


In [ ]:

# Zadržavamo autore sa najmanje 20 komentara
eligible_authors = comments_per_author[
    comments_per_author >= 20
].index

filtered_df = merged_df[
    merged_df["author"].isin(eligible_authors)
].copy()

# Najviše 100 nasumično odabranih komentara po autoru
sampled_df = (
    filtered_df
    .groupby("author", group_keys=False)
    .sample(n=100, replace=False, random_state=42)
    if False else None
)

# Uzorkovanje bez ponavljanja, uz različite veličine grupa
sampled_df = (
    filtered_df
    .groupby("author", group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(len(group), 100),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

# Spajamo komentare u jedan tekst po autoru
author_texts = (
    sampled_df.groupby("author")["body"]
    .agg(" ".join)
    .reset_index(name="text")
)

# Dodajemo Big Five oznake
author_df = author_texts.merge(
    selected_profiles,
    on="author",
    how="inner",
    validate="one_to_one"
)

print("Broj autora:", len(author_df))
print("Jedinstveni autori:", author_df["author"].nunique())
print("Broj komentara korištenih za tekstove:", len(sampled_df))
print("Prosječan broj riječi po autoru:",
      round(author_df["text"].str.split().str.len().mean(), 1))


### Create the held-out author split

Split the author-level records into training and test partitions using seed 42. No author is shared across partitions.


In [ ]:

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    author_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Broj autora za treniranje:", len(train_df))
print("Broj autora za testiranje:", len(test_df))

overlap = set(train_df["author"]) & set(test_df["author"])

print("Autori prisutni u oba skupa:", len(overlap))

assert len(overlap) == 0, "Pronađeno preklapanje autora!"


### Set up TF–IDF features

Define the lexical representation used by the primary regression experiments.


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

X_train_text = train_df["text"]
X_test_text = test_df["text"]

y_train = train_df[traits]
y_test = test_df[traits]

# Definišemo TF-IDF konfiguraciju
tfidf = TfidfVectorizer(
    max_features=10000,
    min_df=3,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

print("Broj trening tekstova:", len(X_train_text))
print("Broj test tekstova:", len(X_test_text))
print("Dimenzije trening oznaka:", y_train.shape)
print("Dimenzije test oznaka:", y_test.shape)
print("TF-IDF konfiguracija je spremna.")


### Define Ridge Regression and cross-validation

Prepare the Ridge pipeline and the training-only hyperparameter search.


In [ ]:

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error

# TF-IDF + Ridge u jednom pipelineu
ridge_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("model", Ridge())
])

# Vrijednosti hiperparametra koje testiramo
ridge_params = {
    "model__alpha": [0.1, 1.0, 5.0, 10.0, 20.0]
}

# Svaki autor mora ostati u jednom foldu
group_cv = GroupKFold(n_splits=5)

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_params,
    scoring="neg_mean_absolute_error",
    cv=group_cv,
    n_jobs=-1,
    refit=True
)

# Za početak treniramo samo jednu osobinu: agreeableness
ridge_search.fit(
    X_train_text,
    y_train["agreeableness"],
    groups=train_df["author"]
)

print("Najbolji alpha:", ridge_search.best_params_)
print("Najbolji CV MAE:", round(-ridge_search.best_score_, 4))


### Evaluate the mean baseline

Estimate performance from a constant predictor before comparing trained regressors.


In [ ]:

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score

baseline_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("model", DummyRegressor(strategy="mean"))
])

baseline_scores = cross_val_score(
    baseline_pipeline,
    X_train_text,
    y_train["agreeableness"],
    groups=train_df["author"],
    cv=group_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

baseline_mae = -baseline_scores.mean()
ridge_mae = -ridge_search.best_score_

print("Baseline CV MAE:", round(baseline_mae, 4))
print("Ridge CV MAE:", round(ridge_mae, 4))
print("Razlika (Baseline - Ridge):", round(baseline_mae - ridge_mae, 4))


### Tune Ridge Regression for each trait

Search regularization strengths independently for the five Big Five targets.


In [ ]:

ridge_results = []

for trait in traits:
    print(f"\nTrenutno obrađujemo: {trait}")

    # Odabir najboljeg alpha parametra
    search = GridSearchCV(
        estimator=ridge_pipeline,
        param_grid=ridge_params,
        scoring="neg_mean_absolute_error",
        cv=group_cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(
        X_train_text,
        y_train[trait],
        groups=train_df["author"]
    )

    # Baseline evaluacija
    baseline_scores = cross_val_score(
        DummyRegressor(strategy="mean"),
        X_train_text,
        y_train[trait],
        groups=train_df["author"],
        cv=group_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    baseline_mae = -baseline_scores.mean()
    ridge_mae = -search.best_score_

    ridge_results.append({
        "Trait": trait,
        "Best Alpha": search.best_params_["model__alpha"],
        "Ridge CV MAE": round(ridge_mae, 4),
        "Baseline CV MAE": round(baseline_mae, 4),
        "Improvement (%)": round(
            (baseline_mae - ridge_mae) / baseline_mae * 100, 2
        )
    })

ridge_results_df = pd.DataFrame(ridge_results)

print("\nREZULTATI RIDGE REGRESIJE:")
print(ridge_results_df.to_string(index=False))


### Tune LinearSVR

Run the corresponding training-fold search for the linear support vector regressor.


In [ ]:

from sklearn.svm import LinearSVR

svr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("model", LinearSVR(
        random_state=42,
        max_iter=10000,
        dual=True
    ))
])

svr_params = {
    "model__C": [0.01, 0.1, 1.0, 10.0]
}

svr_results = []

for trait in traits:
    print(f"\nTrenutno obrađujemo: {trait}")

    search = GridSearchCV(
        estimator=svr_pipeline,
        param_grid=svr_params,
        scoring="neg_mean_absolute_error",
        cv=group_cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(
        X_train_text,
        y_train[trait],
        groups=train_df["author"]
    )

    svr_results.append({
        "Trait": trait,
        "Best C": search.best_params_["model__C"],
        "LinearSVR CV MAE": round(-search.best_score_, 4)
    })

svr_results_df = pd.DataFrame(svr_results)

print("\nREZULTATI LINEARSVR MODELA:")
print(svr_results_df.to_string(index=False))


### Tune SGDRegressor

Evaluate the SGD-based linear regressor using the same author-level data split.


In [ ]:

from sklearn.linear_model import SGDRegressor

sgd_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("model", SGDRegressor(
        loss="squared_error",
        penalty="l2",
        max_iter=3000,
        tol=1e-3,
        random_state=42
    ))
])

sgd_params = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1]
}

sgd_results = []

for trait in traits:
    print(f"\nTrenutno obrađujemo: {trait}")

    search = GridSearchCV(
        estimator=sgd_pipeline,
        param_grid=sgd_params,
        scoring="neg_mean_absolute_error",
        cv=group_cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(
        X_train_text,
        y_train[trait],
        groups=train_df["author"]
    )

    sgd_results.append({
        "Trait": trait,
        "Best Alpha": search.best_params_["model__alpha"],
        "SGD CV MAE": round(-search.best_score_, 4)
    })

sgd_results_df = pd.DataFrame(sgd_results)

print("\nREZULTATI SGD REGRESIJE:")
print(sgd_results_df.to_string(index=False))


### Tune XGBoost

Evaluate the tree-based regressor with its smaller TF–IDF feature budget.


In [ ]:

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Manje zahtjevna konfiguracija
xgb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=3000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_estimators=100,
        max_depth=3,
        random_state=42,
        n_jobs=1
    ))
])

# Manji broj kombinacija hiperparametara
xgb_params = {
    "model__learning_rate": [0.05, 0.1]
}

xgb_results = []

for trait in traits:
    print(f"\nTrenutno obrađujemo: {trait}", flush=True)

    search = GridSearchCV(
        estimator=xgb_pipeline,
        param_grid=xgb_params,
        scoring="neg_mean_absolute_error",
        cv=group_cv,
        n_jobs=1,
        refit=True
    )

    search.fit(
        X_train_text,
        y_train[trait],
        groups=train_df["author"]
    )

    xgb_results.append({
        "Trait": trait,
        "Best Learning Rate": search.best_params_["model__learning_rate"],
        "XGBoost CV MAE": round(-search.best_score_, 4)
    })

    print(
        f"Završeno: {trait}, "
        f"MAE = {-search.best_score_:.4f}",
        flush=True
    )

xgb_results_df = pd.DataFrame(xgb_results)

print("\nREZULTATI XGBOOST MODELA:")
print(xgb_results_df.to_string(index=False))


### Inspect XGBoost search output

Check that the five trait-specific searches finished and returned results.


In [ ]:

print("Broj završenih osobina:", len(xgb_results))

for result in xgb_results:
    print(result)


### Collect selected hyperparameters

Combine the cross-validation choices for the four model families.


In [ ]:

best_params = {
    "Ridge": dict(zip(
        ridge_results_df["Trait"],
        ridge_results_df["Best Alpha"]
    )),
    "LinearSVR": dict(zip(
        svr_results_df["Trait"],
        svr_results_df["Best C"]
    )),
    "SGD": dict(zip(
        sgd_results_df["Trait"],
        sgd_results_df["Best Alpha"]
    )),
    "XGBoost": dict(zip(
        xgb_results_df["Trait"],
        xgb_results_df["Best Learning Rate"]
    ))
}

for model, parameters in best_params.items():
    print(f"\n{model}:")
    for trait, value in parameters.items():
        print(f"  {trait}: {value}")


### Evaluate Ridge on unseen authors

Fit the selected Ridge configurations and calculate held-out test metrics.


In [ ]:

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

ridge_test_results = []

for trait in traits:
    # Ridge s prethodno odabranom alphom
    model = clone(ridge_pipeline)
    model.set_params(
        model__alpha=best_params["Ridge"][trait]
    )

    model.fit(X_train_text, y_train[trait])
    predictions = model.predict(X_test_text)

    # Baseline: prosjek iz trening skupa
    baseline = DummyRegressor(strategy="mean")
    baseline.fit(
        np.zeros((len(y_train), 1)),
        y_train[trait]
    )
    baseline_predictions = baseline.predict(
        np.zeros((len(y_test), 1))
    )

    ridge_test_results.append({
        "Trait": trait,
        "MAE": round(
            mean_absolute_error(y_test[trait], predictions), 4
        ),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_test[trait], predictions)), 4
        ),
        "R2": round(
            r2_score(y_test[trait], predictions), 4
        ),
        "Baseline MAE": round(
            mean_absolute_error(y_test[trait], baseline_predictions), 4
        )
    })

ridge_test_df = pd.DataFrame(ridge_test_results)

print("RIDGE — ZAVRŠNI TEST REZULTATI:")
print(ridge_test_df.to_string(index=False))


### Evaluate LinearSVR on unseen authors

Use the training-selected values of `C` for each trait.


In [ ]:

svr_test_results = []

for trait in traits:
    model = clone(svr_pipeline)

    model.set_params(
        model__C=best_params["LinearSVR"][trait]
    )

    model.fit(X_train_text, y_train[trait])
    predictions = model.predict(X_test_text)

    svr_test_results.append({
        "Trait": trait,
        "MAE": round(
            mean_absolute_error(y_test[trait], predictions), 4
        ),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_test[trait], predictions)), 4
        ),
        "R2": round(
            r2_score(y_test[trait], predictions), 4
        )
    })

svr_test_df = pd.DataFrame(svr_test_results)

print("LINEARSVR — ZAVRŠNI TEST REZULTATI:")
print(svr_test_df.to_string(index=False))


### Evaluate SGDRegressor on unseen authors

Apply the regularization values chosen during training cross-validation.


In [ ]:

sgd_test_results = []

for trait in traits:
    model = clone(sgd_pipeline)

    # Koristimo alpha odabranu tokom validacije
    model.set_params(
        model__alpha=best_params["SGD"][trait]
    )

    # Treniranje samo na trening autorima
    model.fit(X_train_text, y_train[trait])

    # Predviđanje za 274 test autora
    predictions = model.predict(X_test_text)

    sgd_test_results.append({
        "Trait": trait,
        "MAE": round(
            mean_absolute_error(y_test[trait], predictions), 4
        ),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_test[trait], predictions)), 4
        ),
        "R2": round(
            r2_score(y_test[trait], predictions), 4
        )
    })

sgd_test_df = pd.DataFrame(sgd_test_results)

print("SGD — ZAVRŠNI TEST REZULTATI:")
print(sgd_test_df.to_string(index=False))


### Evaluate XGBoost on unseen authors

Use the selected learning rates and record held-out test metrics.


In [ ]:

xgb_test_results = []

for trait in traits:
    print(f"Testiramo: {trait}", flush=True)

    model = clone(xgb_pipeline)

    # Najbolji learning rate iz validacije
    model.set_params(
        model__learning_rate=best_params["XGBoost"][trait]
    )

    # Treniranje na 1.094 trening autora
    model.fit(X_train_text, y_train[trait])

    # Evaluacija na 274 test autora
    predictions = model.predict(X_test_text)

    xgb_test_results.append({
        "Trait": trait,
        "MAE": round(
            mean_absolute_error(y_test[trait], predictions), 4
        ),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_test[trait], predictions)), 4
        ),
        "R2": round(
            r2_score(y_test[trait], predictions), 4
        )
    })

    print(f"Završeno: {trait}", flush=True)

xgb_test_df = pd.DataFrame(xgb_test_results)

print("\nXGBOOST — ZAVRŠNI TEST REZULTATI:")
print(xgb_test_df.to_string(index=False))


### Save model-selection and test tables

Export the experiment tables to the `results` folder on Google Drive.


In [ ]:

import os
import json
import pandas as pd

save_dir = "/content/drive/MyDrive/PANDORA_Original/results"
os.makedirs(save_dir, exist_ok=True)

# 1. Rezultati unakrsne validacije
cv_tables = {
    "ridge_cv": ridge_results_df,
    "linearsvr_cv": svr_results_df,
    "sgd_cv": sgd_results_df,
    "xgboost_cv": xgb_results_df
}

# 2. Završni test rezultati
test_tables = {
    "ridge_test": ridge_test_df,
    "linearsvr_test": svr_test_df,
    "sgd_test": sgd_test_df,
    "xgboost_test": xgb_test_df
}

for name, df in {**cv_tables, **test_tables}.items():
    df.to_csv(f"{save_dir}/{name}.csv", index=False)

# 3. Najbolji hiperparametri
best_params_clean = {
    model: {
        trait: float(value)
        for trait, value in params.items()
    }
    for model, params in best_params.items()
}

with open(f"{save_dir}/best_params.json", "w") as f:
    json.dump(best_params_clean, f, indent=4)

# 4. Pripremljeni podaci i fiksna podjela autora
train_df.to_pickle(f"{save_dir}/train_authors.pkl")
test_df.to_pickle(f"{save_dir}/test_authors.pkl")

print("Svi rezultati, hiperparametri i podaci su sačuvani!")
print("Lokacija:", save_dir)
print("Broj sačuvanih datoteka:", len(os.listdir(save_dir)))


### Save trained model artifacts

Persist fitted pipelines for later checks that do not require retraining.


In [ ]:

import joblib
import os
from sklearn.base import clone

models_dir = f"{save_dir}/trained_models"
os.makedirs(models_dir, exist_ok=True)

model_configs = {
    "Ridge": (ridge_pipeline, "model__alpha"),
    "LinearSVR": (svr_pipeline, "model__C"),
    "SGD": (sgd_pipeline, "model__alpha"),
    "XGBoost": (xgb_pipeline, "model__learning_rate")
}

for model_name, (pipeline, param_name) in model_configs.items():
    for trait in traits:
        print(f"Treniramo i spremamo: {model_name} - {trait}", flush=True)

        model = clone(pipeline)
        model.set_params(
            **{param_name: best_params[model_name][trait]}
        )

        model.fit(X_train_text, y_train[trait])

        file_path = f"{models_dir}/{model_name}_{trait}.joblib"
        joblib.dump(model, file_path)

        print(f"Sačuvano: {model_name} - {trait}", flush=True)

print("\nUkupno sačuvanih modela:",
      len([f for f in os.listdir(models_dir) if f.endswith(".joblib")]))


### Record software and run metadata

Write out environment and configuration information captured during this run. Versions may differ from a later Colab session.


In [ ]:

import sys
import sklearn
import pandas as pd
import numpy as np
import xgboost
import joblib
import json
import os

# 1. Spremanje verzija biblioteka
versions = {
    "python": sys.version.split()[0],
    "scikit-learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "xgboost": xgboost.__version__,
    "joblib": joblib.__version__
}

with open(f"{save_dir}/environment_versions.json", "w") as f:
    json.dump(versions, f, indent=4)

# 2. Objedinjena tabela završnih rezultata
model_tables = {
    "Ridge": ridge_test_df,
    "LinearSVR": svr_test_df,
    "SGDRegressor": sgd_test_df,
    "XGBoost": xgb_test_df
}

all_results = []

for model_name, df in model_tables.items():
    temp = df.copy()
    temp["Model"] = model_name
    all_results.append(temp)

final_results_df = pd.concat(all_results, ignore_index=True)

final_results_df.to_csv(
    f"{save_dir}/all_models_test_results.csv",
    index=False
)

print("Verzije biblioteka:")
print(versions)

print("\nUkupan broj rezultata:", len(final_results_df))
print("Sve je sačuvano na Google Drive!")


### Summarize the five target distributions

Calculate descriptive statistics on the retained author-level labels.


In [ ]:

import pandas as pd

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

trait_stats = author_df[traits].agg(["mean", "std"]).T
trait_stats.columns = ["Mean", "SD"]
trait_stats.index.name = "Personality Trait"

print("Number of authors:", len(author_df))
print("\nBig Five descriptive statistics:")
display(trait_stats.round(4))


### Summarize available comment counts

Measure how much source text was available before applying the 100-comment cap.


In [ ]:

# Number of original comments per retained author
comment_counts = (
    clean_comments[
        clean_comments["author"].isin(author_df["author"])
    ]
    .groupby("author")
    .size()
)

print("Number of authors:", len(comment_counts))
print("Minimum comments:", comment_counts.min())
print("Median comments:", comment_counts.median())
print("Mean comments:", round(comment_counts.mean(), 2))
print("Maximum comments:", comment_counts.max())


## Figures from the held-out evaluation

The following two plots use the recorded MAE and RMSE values. They are visual summaries, not additional model fitting.


### Plot held-out MAE

Create the MAE comparison from the recorded test results.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

traits = [
    "Agreeableness", "Openness", "Conscientiousness",
    "Extraversion", "Neuroticism"
]

mae_data = pd.DataFrame({
    "Ridge": [0.2646, 0.2318, 0.2649, 0.2582, 0.2813],
    "LinearSVR": [0.2643, 0.2314, 0.2614, 0.2611, 0.2829],
    "SGD": [0.2724, 0.2375, 0.2660, 0.2576, 0.2891],
    "XGBoost": [0.2685, 0.2335, 0.2709, 0.2549, 0.2844],
    "Mean Baseline": [0.2699, 0.2346, 0.2667, 0.2595, 0.2843]
}, index=traits)

ax = mae_data.plot(kind="bar", figsize=(12, 6), width=0.8)

ax.set_title("Held-out Test MAE by Model and Personality Trait")
ax.set_xlabel("Personality Trait")
ax.set_ylabel("Mean Absolute Error (MAE)")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Model")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig("figure_2_mae.png", dpi=300, bbox_inches="tight")
plt.show()


### Plot held-out RMSE

Create the corresponding RMSE figure from the recorded test results.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

traits = [
    "Agreeableness", "Openness", "Conscientiousness",
    "Extraversion", "Neuroticism"
]

# Recorded final held-out test RMSE values (274 authors)
rmse_data = pd.DataFrame({
    "Ridge": [0.3019, 0.2760, 0.3110, 0.3059, 0.3245],
    "LinearSVR": [0.3056, 0.2830, 0.3135, 0.3109, 0.3288],
    "SGD": [0.3174, 0.2833, 0.3128, 0.3242, 0.3322],
    "XGBoost": [0.3064, 0.2767, 0.3140, 0.2957, 0.3243]
}, index=traits)

ax = rmse_data.plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8
)

ax.set_title("Held-out Test RMSE by Model and Personality Trait")
ax.set_xlabel("Personality Trait")
ax.set_ylabel("Root Mean Squared Error (RMSE)")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Model")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig("figure_3_rmse.png", dpi=300, bbox_inches="tight")
plt.show()


## Additional evaluation and revision analyses

The remaining cells extend the initial evaluation with Pearson correlations, a median baseline, uncertainty estimates, expanded tuning checks, contextual embeddings, and two ablations. The code follows the order in which the checks were carried out; some cells reload files or reconnect Drive after a runtime restart.


### Calculate Pearson correlations

Load saved models and compute the association between predicted and observed trait scores.


In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os
import joblib
import pandas as pd
from scipy.stats import pearsonr

results_dir = "/content/drive/MyDrive/PANDORA_Original/results"
models_dir = os.path.join(results_dir, "trained_models")

test_df = pd.read_pickle(
    os.path.join(results_dir, "test_authors.pkl")
)

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

models = ["Ridge", "LinearSVR", "SGD", "XGBoost"]

rows = []

for model_name in models:
    for trait in traits:
        model_path = os.path.join(
            models_dir, f"{model_name}_{trait}.joblib"
        )

        model = joblib.load(model_path)

        y_true = test_df[trait].to_numpy()
        y_pred = model.predict(test_df["text"])

        r, p_value = pearsonr(y_true, y_pred)

        rows.append({
            "Model": model_name,
            "Trait": trait,
            "Pearson_r": r
        })

pearson_df = pd.DataFrame(rows)

print(pearson_df.to_string(index=False))

pearson_df.to_csv(
    os.path.join(results_dir, "pearson_test_results.csv"),
    index=False
)

print("\nPearson rezultati su sačuvani na Google Drive!")


### Evaluate the median baseline

Compare the selected models with a constant training-median predictor.


In [ ]:

import os
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

drive.mount("/content/drive")

results_dir = "/content/drive/MyDrive/PANDORA_Original/results"

# Load original saved training and test DataFrames
train_df = pd.read_pickle(
    os.path.join(results_dir, "train_authors.pkl")
)
test_df = pd.read_pickle(
    os.path.join(results_dir, "test_authors.pkl")
)

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

# Verify original split and label scale
print("Training authors:", len(train_df))
print("Test authors:", len(test_df))

assert len(train_df) == 1094
assert len(test_df) == 274
assert set(train_df["author"]).isdisjoint(set(test_df["author"]))
assert train_df[traits].notna().all().all()
assert test_df[traits].notna().all().all()

all_labels = pd.concat([train_df[traits], test_df[traits]])
assert all_labels.min().min() >= 0
assert all_labels.max().max() <= 1

# Calculate Median Baseline on held-out test set
results = []

for trait in traits:
    y_train = train_df[trait].to_numpy()
    y_test = test_df[trait].to_numpy()

    model = DummyRegressor(strategy="median")
    model.fit(np.zeros((len(y_train), 1)), y_train)

    predictions = model.predict(
        np.zeros((len(y_test), 1))
    )

    results.append({
        "Trait": trait,
        "Training Median": np.median(y_train),
        "Median Baseline MAE": mean_absolute_error(
            y_test, predictions
        ),
        "Median Baseline RMSE": np.sqrt(
            mean_squared_error(y_test, predictions)
        )
    })

median_results = pd.DataFrame(results)

print("\nMEDIAN BASELINE — ORIGINAL TEST SET")
print(median_results.round(4).to_string(index=False))


### Export the median-baseline results

Save the additional baseline table to the same results directory.


In [ ]:
median_results.to_csv(
    "/content/drive/MyDrive/PANDORA_Original/results/median_baseline_test.csv",
    index=False
)
print("Median Baseline results saved.")

### Reload the saved train and test authors

Restore the author-level partitions for the later uncertainty analysis.


In [ ]:

import os
import pandas as pd

results_dir = "/content/drive/MyDrive/PANDORA_Original/results"
models_dir = os.path.join(results_dir, "trained_models")

train_df = pd.read_pickle(
    os.path.join(results_dir, "train_authors.pkl")
)
test_df = pd.read_pickle(
    os.path.join(results_dir, "test_authors.pkl")
)

model_files = (
    sorted(f for f in os.listdir(models_dir)
           if f.endswith(".joblib"))
    if os.path.isdir(models_dir) else []
)

print("Training authors:", len(train_df))
print("Test authors:", len(test_df))
print("Saved models:", len(model_files))
print("\nModel files:")
for filename in model_files:
    print(filename)

print("\nTest columns:", test_df.columns.tolist())


### Bootstrap intervals and paired comparisons

Use held-out predictions to estimate uncertainty and compare model errors with the baselines.


In [ ]:

import os
import joblib
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, wilcoxon

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"
MODELS = os.path.join(RESULTS, "trained_models")
B = 2000
rng = np.random.default_rng(42)

train = pd.read_pickle(os.path.join(RESULTS, "train_authors.pkl"))
test = pd.read_pickle(os.path.join(RESULTS, "test_authors.pkl"))

selected = {
    "agreeableness": "Ridge",
    "openness": "LinearSVR",
    "conscientiousness": "LinearSVR",
    "extraversion": "Ridge",
    "neuroticism": "Ridge"
}

def safe_pearson(y, p):
    if np.std(y) == 0 or np.std(p) == 0:
        return np.nan
    return float(pearsonr(y, p).statistic)

def interval(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return (
        float(np.percentile(values, 2.5)),
        float(np.percentile(values, 97.5))
    ) if len(values) else (np.nan, np.nan)

metric_rows = []
comparison_rows = []

for trait, model_name in selected.items():
    model = joblib.load(
        os.path.join(MODELS, f"{model_name}_{trait}.joblib")
    )

    y = test[trait].to_numpy(dtype=float)
    pred = np.asarray(model.predict(test["text"]), dtype=float)

    baselines = {
        "Mean Baseline": np.full(len(y), train[trait].mean()),
        "Median Baseline": np.full(len(y), train[trait].median())
    }

    n = len(y)
    indices = rng.integers(0, n, size=(B, n))

    # Bootstrap MAE
    abs_error = np.abs(y - pred)
    boot_mae = abs_error[indices].mean(axis=1)

    # Bootstrap Pearson r
    boot_r = np.array([
        safe_pearson(y[idx], pred[idx])
        for idx in indices
    ])

    mae_low, mae_high = interval(boot_mae)
    r_low, r_high = interval(boot_r)

    metric_rows.append({
        "Trait": trait,
        "Model": model_name,
        "MAE": abs_error.mean(),
        "MAE_CI_Low": mae_low,
        "MAE_CI_High": mae_high,
        "Pearson_r": safe_pearson(y, pred),
        "Pearson_CI_Low": r_low,
        "Pearson_CI_High": r_high
    })

    # Paired comparison with both baselines
    for baseline_name, baseline_pred in baselines.items():
        baseline_error = np.abs(y - baseline_pred)

        # Positive difference means selected model is better
        difference = baseline_error - abs_error
        boot_diff = difference[indices].mean(axis=1)
        diff_low, diff_high = interval(boot_diff)

        try:
            w_p = float(wilcoxon(
                difference,
                alternative="two-sided"
            ).pvalue)
        except ValueError:
            w_p = np.nan

        comparison_rows.append({
            "Trait": trait,
            "Model": model_name,
            "Baseline": baseline_name,
            "Model_MAE": abs_error.mean(),
            "Baseline_MAE": baseline_error.mean(),
            "MAE_Improvement": difference.mean(),
            "Improvement_CI_Low": diff_low,
            "Improvement_CI_High": diff_high,
            "Wilcoxon_p": w_p
        })

metric_results = pd.DataFrame(metric_rows)
comparison_results = pd.DataFrame(comparison_rows)

print("\n95% BOOTSTRAP CONFIDENCE INTERVALS")
print(metric_results.round(4).to_string(index=False))

print("\nPAIRED BASELINE COMPARISONS")
print(comparison_results.round(4).to_string(index=False))

metric_results.to_csv(
    os.path.join(RESULTS, "bootstrap_model_ci.csv"),
    index=False
)
comparison_results.to_csv(
    os.path.join(RESULTS, "bootstrap_baseline_comparisons.csv"),
    index=False
)

print("\nBoth CSV files saved successfully.")


### Extend the Ridge regularization search

Revisit Ridge alpha values on the training partition, without selecting parameters from test scores.


In [ ]:

import os
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

train_df = pd.read_pickle(
    os.path.join(RESULTS, "train_authors.pkl")
)

assert len(train_df) == 1094
assert train_df["author"].nunique() == 1094

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

alphas = [
    0.001, 0.01, 0.05, 0.1,
    0.5, 1, 2, 5, 10, 20, 50, 100
]

# Prepare the five author-disjoint folds
cv = GroupKFold(n_splits=5)
fold_data = []

for fold_num, (train_idx, val_idx) in enumerate(
    cv.split(
        train_df["text"],
        groups=train_df["author"]
    ),
    start=1
):
    print(f"Preparing TF-IDF fold {fold_num}/5", flush=True)

    vectorizer = TfidfVectorizer(
        max_features=10000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(
        train_df.iloc[train_idx]["text"]
    )
    X_val = vectorizer.transform(
        train_df.iloc[val_idx]["text"]
    )

    fold_data.append(
        (train_idx, val_idx, X_train, X_val)
    )

print("TF-IDF preparation completed.", flush=True)

# Tune Ridge using cached fold matrices
rows = []

for trait in traits:
    print(f"\nTuning Ridge: {trait}", flush=True)

    y = train_df[trait].to_numpy()

    for alpha in alphas:
        fold_maes = []

        for train_idx, val_idx, X_train, X_val in fold_data:
            model = Ridge(alpha=alpha)

            model.fit(X_train, y[train_idx])
            predictions = model.predict(X_val)

            fold_maes.append(
                mean_absolute_error(
                    y[val_idx], predictions
                )
            )

        rows.append({
            "Trait": trait,
            "Alpha": alpha,
            "CV MAE": np.mean(fold_maes),
            "CV MAE SD": np.std(fold_maes)
        })

    print(f"Finished: {trait}", flush=True)

grid_results = pd.DataFrame(rows)

best_results = (
    grid_results
    .sort_values(["Trait", "CV MAE", "Alpha"])
    .groupby("Trait", as_index=False)
    .first()
)

best_results["At Grid Boundary"] = (
    best_results["Alpha"].isin([
        min(alphas), max(alphas)
    ])
)

print("\nEXTENDED RIDGE TUNING RESULTS")
print(best_results.round(4).to_string(index=False))

# Save as new files; original results stay untouched
grid_results.to_csv(
    os.path.join(RESULTS, "ridge_extended_grid_details.csv"),
    index=False
)

best_results.to_csv(
    os.path.join(RESULTS, "ridge_extended_tuning.csv"),
    index=False
)

print("\nExtended Ridge tuning saved successfully.")


### Inspect SGD regularization near zero

Check whether very small alpha values alter the cross-validation error.


In [ ]:

from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd
import os

# Reuse the five TF-IDF folds from Ridge tuning.
assert len(fold_data) == 5, "First run the Ridge preparation cell."

sgd_alphas = [
    0.000001, 0.00001, 0.00005,
    0.0001, 0.0005, 0.001,
    0.005, 0.01, 0.05, 0.1, 0.5, 1.0
]

sgd_rows = []

for trait in traits:
    print(f"\nTuning SGD: {trait}", flush=True)
    y = train_df[trait].to_numpy()

    for alpha in sgd_alphas:
        fold_maes = []

        for train_idx, val_idx, X_train, X_val in fold_data:
            model = SGDRegressor(
                alpha=alpha,
                loss="squared_error",
                penalty="l2",
                max_iter=2000,
                tol=1e-3,
                random_state=42
            )

            model.fit(X_train, y[train_idx])
            predictions = model.predict(X_val)

            fold_maes.append(
                mean_absolute_error(y[val_idx], predictions)
            )

        sgd_rows.append({
            "Trait": trait,
            "Alpha": alpha,
            "CV MAE": np.mean(fold_maes),
            "CV MAE SD": np.std(fold_maes)
        })

    print(f"Finished: {trait}", flush=True)

sgd_grid = pd.DataFrame(sgd_rows)

sgd_best = (
    sgd_grid
    .sort_values(["Trait", "CV MAE", "Alpha"])
    .groupby("Trait", as_index=False)
    .first()
)

sgd_best["At Grid Boundary"] = (
    sgd_best["Alpha"].isin([
        min(sgd_alphas), max(sgd_alphas)
    ])
)

print("\nEXTENDED SGD TUNING RESULTS")
print(sgd_best.round(6).to_string(index=False))

sgd_grid.to_csv(
    os.path.join(RESULTS, "sgd_extended_grid_details.csv"),
    index=False
)

sgd_best.to_csv(
    os.path.join(RESULTS, "sgd_extended_tuning.csv"),
    index=False
)

print("\nExtended SGD tuning saved successfully.")


### Extend the small-alpha check

Evaluate additional SGD alpha values for Openness and Neuroticism.


In [ ]:

extra_alphas = [
    1e-9, 1e-8, 1e-7, 1e-6,
    1e-5, 1e-4
]

extra_rows = []

for trait in ["openness", "neuroticism"]:
    print(f"\nChecking lower alpha values: {trait}", flush=True)

    y = train_df[trait].to_numpy()

    for alpha in extra_alphas:
        fold_maes = []

        for train_idx, val_idx, X_train, X_val in fold_data:
            model = SGDRegressor(
                alpha=alpha,
                loss="squared_error",
                penalty="l2",
                max_iter=2000,
                tol=1e-3,
                random_state=42
            )

            model.fit(X_train, y[train_idx])
            pred = model.predict(X_val)

            fold_maes.append(
                mean_absolute_error(y[val_idx], pred)
            )

        extra_rows.append({
            "Trait": trait,
            "Alpha": alpha,
            "CV MAE": np.mean(fold_maes)
        })

extra_df = pd.DataFrame(extra_rows)

combined_sgd = pd.concat(
    [sgd_grid, extra_df],
    ignore_index=True
)

best_two = (
    combined_sgd[
        combined_sgd["Trait"].isin(
            ["openness", "neuroticism"]
        )
    ]
    .sort_values(["Trait", "CV MAE"])
    .groupby("Trait", as_index=False)
    .first()
)

print("\nLOWER-ALPHA SGD RESULTS")
print(best_two.to_string(index=False))

combined_sgd.to_csv(
    os.path.join(RESULTS, "sgd_extended_grid_details.csv"),
    index=False
)

print("\nAdditional SGD results saved.")


### Inspect saved XGBoost configurations

Check the available model files and recorded settings before the extended search.


In [ ]:

import json
import os
import joblib

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"
MODELS = os.path.join(RESULTS, "trained_models")

print("XGBOOST MODEL FILES:")
for filename in sorted(os.listdir(MODELS)):
    if "xgboost" in filename.lower() or "xgb" in filename.lower():
        print(filename)

print("\nSAVED BEST PARAMETERS:")
with open(os.path.join(RESULTS, "best_params.json")) as f:
    best_params = json.load(f)

print(json.dumps(best_params.get("XGBoost", {}), indent=2))

print("\nSAVED XGBOOST PIPELINE:")
files = [
    f for f in os.listdir(MODELS)
    if "xgboost" in f.lower() or "xgb" in f.lower()
]

if files:
    model = joblib.load(os.path.join(MODELS, files[0]))
    print("Model type:", type(model).__name__)

    if hasattr(model, "named_steps"):
        for name, step in model.named_steps.items():
            print(f"\n{name}: {type(step).__name__}")
            if hasattr(step, "get_params"):
                print(step.get_params())
    elif hasattr(model, "get_params"):
        print(model.get_params())
else:
    print("No matching XGBoost model file found.")


### Test additional XGBoost tree counts

Evaluate estimator counts using the previously prepared TF–IDF folds.


In [ ]:

import time
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

assert len(xgb_fold_data) == 5, (
    "XGBoost TF-IDF foldovi nisu pripremljeni."
)

# Test only one trait, one fold, one learning rate
trait = "agreeableness"
learning_rate = 0.05

tr_idx, val_idx, X_tr, X_val = xgb_fold_data[0]
y = train_df[trait].to_numpy()

print("Starting XGBoost speed test...", flush=True)

start = time.perf_counter()

model = XGBRegressor(
    objective="reg:squarederror",
    learning_rate=learning_rate,
    max_depth=3,
    n_estimators=100,
    tree_method="hist",
    n_jobs=1,
    random_state=42
)

model.fit(X_tr, y[tr_idx])
pred = model.predict(X_val)

elapsed = time.perf_counter() - start
mae = mean_absolute_error(y[val_idx], pred)

print(f"\nOne training took: {elapsed:.2f} seconds")
print(f"Validation MAE: {mae:.4f}")
print(f"Estimated 150 trainings: {elapsed * 150 / 60:.1f} minutes")


### Inspect runtime and fold dimensions

Record CPU availability, XGBoost version, and matrix size for the follow-up tests.


In [ ]:

import os
import xgboost

print("Available CPU cores:", os.cpu_count())
print("XGBoost version:", xgboost.__version__)
print("Training matrix shape:", xgb_fold_data[0][2].shape)
print("Training matrix type:", type(xgb_fold_data[0][2]).__name__)


### Time a single XGBoost fold

Measure the cost of the next search before running the larger parameter combinations.


In [ ]:

import time
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

tr_idx, val_idx, X_tr, X_val = xgb_fold_data[0]

X_tr_fast = X_tr.astype(np.float32)
X_val_fast = X_val.astype(np.float32)

y = train_df["agreeableness"].to_numpy()

print("Starting float32 XGBoost speed test...", flush=True)

start = time.perf_counter()

fast_model = XGBRegressor(
    objective="reg:squarederror",
    learning_rate=0.05,
    max_depth=3,
    n_estimators=100,
    tree_method="hist",
    n_jobs=2,
    random_state=42
)

fast_model.fit(X_tr_fast, y[tr_idx])
pred = fast_model.predict(X_val_fast)

elapsed = time.perf_counter() - start
mae = mean_absolute_error(y[val_idx], pred)

print(f"\nTraining time: {elapsed:.2f} seconds")
print(f"Validation MAE: {mae:.4f}")
print(f"Previous MAE: 0.2540")
print(f"Estimated 150 trainings: {elapsed * 150 / 60:.1f} minutes")


### Inspect additional Ridge alpha values

Check the expanded regularization range for the relevant traits.


In [ ]:

import numpy as np
import pandas as pd
import os
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

# Additional alpha values requested by the professor
extra_alphas = [100, 1000, 10000]
extra_rows = []

for trait in traits:
    print(f"Checking Ridge: {trait}", flush=True)
    y = train_df[trait].to_numpy()

    for alpha in extra_alphas:
        fold_maes = []

        for train_idx, val_idx, X_train, X_val in fold_data:
            model = Ridge(alpha=alpha)
            model.fit(X_train, y[train_idx])

            pred = model.predict(X_val)
            fold_maes.append(
                mean_absolute_error(y[val_idx], pred)
            )

        extra_rows.append({
            "Trait": trait,
            "Alpha": alpha,
            "CV MAE": np.mean(fold_maes),
            "CV MAE SD": np.std(fold_maes)
        })

# Combine with previous extended Ridge results
ridge_full_grid = pd.concat(
    [grid_results, pd.DataFrame(extra_rows)],
    ignore_index=True
).drop_duplicates(subset=["Trait", "Alpha"])

ridge_full_best = (
    ridge_full_grid
    .sort_values(["Trait", "CV MAE", "Alpha"])
    .groupby("Trait", as_index=False)
    .first()
)

ridge_full_best["At Grid Boundary"] = (
    ridge_full_best["Alpha"].isin(
        [ridge_full_grid["Alpha"].min(),
         ridge_full_grid["Alpha"].max()]
    )
)

print("\nFINAL RIDGE TUNING RESULTS")
print(ridge_full_best.round(5).to_string(index=False))

ridge_full_grid.to_csv(
    os.path.join(RESULTS, "ridge_extended_grid_details.csv"),
    index=False
)

ridge_full_best.to_csv(
    os.path.join(RESULTS, "ridge_extended_tuning.csv"),
    index=False
)

print("\nFinal Ridge tuning saved successfully.")


### Time an XGBoost configuration

Benchmark a candidate setting to estimate the runtime of the expanded search.


In [ ]:

import time
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

tr_idx, val_idx, X_tr, X_val = xgb_fold_data[0]

X_tr = X_tr.astype(np.float32)
X_val = X_val.astype(np.float32)
y = train_df["agreeableness"].to_numpy()

configs = [
    {"learning_rate": 0.05, "n_estimators": 100,
     "max_depth": 3, "subsample": 1.0},
    {"learning_rate": 0.05, "n_estimators": 200,
     "max_depth": 3, "subsample": 0.8},
    {"learning_rate": 0.05, "n_estimators": 100,
     "max_depth": 5, "subsample": 0.8},
    {"learning_rate": 0.10, "n_estimators": 200,
     "max_depth": 3, "subsample": 0.8},
]

rows = []

for i, config in enumerate(configs, start=1):
    print(f"Testing configuration {i}/4...", flush=True)
    start = time.perf_counter()

    model = XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=2,
        random_state=42,
        **config
    )

    model.fit(X_tr, y[tr_idx])
    pred = model.predict(X_val)

    rows.append({
        **config,
        "Validation MAE": mean_absolute_error(
            y[val_idx], pred
        ),
        "Seconds": time.perf_counter() - start
    })

pilot_results = pd.DataFrame(rows)

print("\nXGBOOST PILOT RESULTS")
print(pilot_results.round(4).to_string(index=False))

pilot_results.to_csv(
    "/content/drive/MyDrive/PANDORA_Original/results/xgboost_pilot.csv",
    index=False
)

print("\nPilot results saved successfully.")


### Run the expanded XGBoost search

Evaluate the broader estimator, depth, learning-rate, and subsampling combinations.


In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os
import time
import itertools
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"
os.makedirs(RESULTS, exist_ok=True)

train_df = pd.read_pickle(
    os.path.join(RESULTS, "train_authors.pkl")
)

assert len(train_df) == 1094
assert train_df["author"].nunique() == 1094

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

print("XGBoost version:", xgb.__version__)

# Check whether GPU is accessible
import subprocess
gpu_check = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "GPU not detected. Select T4 GPU in Colab runtime settings."
    )

print("GPU detected.")

# Prepare author-disjoint folds
folds = []
cv = GroupKFold(n_splits=5)

for fold_no, (tr, va) in enumerate(
    cv.split(train_df["text"], groups=train_df["author"]),
    start=1
):
    print(f"Preparing TF-IDF fold {fold_no}/5", flush=True)

    vectorizer = TfidfVectorizer(
        max_features=3000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True,
        dtype=np.float32
    )

    Xtr = vectorizer.fit_transform(train_df.iloc[tr]["text"])
    Xva = vectorizer.transform(train_df.iloc[va]["text"])

    folds.append((tr, va, Xtr, Xva))

print("TF-IDF folds ready.", flush=True)

# Systematic grid
grid = list(itertools.product(
    [0.01, 0.05, 0.1],  # learning_rate
    [100, 200],         # n_estimators
    [2, 3, 5],          # max_depth
    [0.7, 1.0]          # subsample
))

print("Configurations:", len(grid))
print("Total model fits:", len(grid) * 5 * len(traits))

output_file = os.path.join(
    RESULTS, "xgboost_gpu_grid_progress.csv"
)

# Resume previous completed combinations
if os.path.exists(output_file):
    rows = pd.read_csv(output_file).to_dict("records")
else:
    rows = []

def key(row):
    return (
        str(row["Trait"]),
        round(float(row["learning_rate"]), 8),
        int(row["n_estimators"]),
        int(row["max_depth"]),
        round(float(row["subsample"]), 8)
    )

completed = {key(r) for r in rows}

for trait in traits:
    y = train_df[trait].to_numpy()

    for lr, trees, depth, subsample in grid:
        config = {
            "Trait": trait,
            "learning_rate": lr,
            "n_estimators": trees,
            "max_depth": depth,
            "subsample": subsample
        }

        if key(config) in completed:
            continue

        print(
            f"{trait}: lr={lr}, trees={trees}, "
            f"depth={depth}, subsample={subsample}",
            flush=True
        )

        fold_maes = []
        start = time.perf_counter()

        for tr, va, Xtr, Xva in folds:
            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                learning_rate=lr,
                n_estimators=trees,
                max_depth=depth,
                subsample=subsample,
                tree_method="hist",
                device="cuda",
                n_jobs=2,
                random_state=42
            )

            model.fit(Xtr, y[tr])
            pred = model.predict(Xva)

            fold_maes.append(
                mean_absolute_error(y[va], pred)
            )

        rows.append({
            **config,
            "CV_MAE": float(np.mean(fold_maes)),
            "CV_MAE_SD": float(np.std(fold_maes)),
            "seconds": time.perf_counter() - start
        })

        pd.DataFrame(rows).to_csv(output_file, index=False)
        completed.add(key(config))

print("\nGRID SEARCH COMPLETED")

results = pd.DataFrame(rows)

best = (
    results.sort_values(["Trait", "CV_MAE"])
    .groupby("Trait", as_index=False)
    .first()
)

print("\nBEST XGBOOST PARAMETERS PER TRAIT")
print(best.to_string(index=False))

best.to_csv(
    os.path.join(RESULTS, "xgboost_gpu_best_params.csv"),
    index=False
)

print("\nAll results saved successfully.")


### Inspect expanded XGBoost results

Summarize and compare the additional training-fold results.


In [ ]:

import os
import time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

# Targeted extensions around boundary solutions
extra_configs = {
    "agreeableness": [
        (0.05, 100, 3, 1.0)
    ],
    "conscientiousness": [
        (0.05, 100, 3, 0.5)
    ],
    "extraversion": [
        (0.10, 300, 5, 0.7),
        (0.10, 200, 7, 0.7),
        (0.20, 200, 5, 0.7)
    ],
    "neuroticism": [
        (0.05, 100, 3, 1.0)
    ],
    "openness": [
        (0.005, 200, 3, 0.7),
        (0.01, 300, 3, 0.7)
    ]
}

rows = []

for trait, configs in extra_configs.items():
    y = train_df[trait].to_numpy()

    for lr, trees, depth, subsample in configs:
        print(
            f"Testing {trait}: lr={lr}, trees={trees}, "
            f"depth={depth}, subsample={subsample}",
            flush=True
        )

        fold_maes = []

        for tr, va, Xtr, Xva in folds:
            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                learning_rate=lr,
                n_estimators=trees,
                max_depth=depth,
                subsample=subsample,
                tree_method="hist",
                device="cuda",
                n_jobs=2,
                random_state=42
            )

            model.fit(Xtr, y[tr])
            pred = model.predict(Xva)

            fold_maes.append(
                mean_absolute_error(y[va], pred)
            )

        rows.append({
            "Trait": trait,
            "learning_rate": lr,
            "n_estimators": trees,
            "max_depth": depth,
            "subsample": subsample,
            "CV_MAE": np.mean(fold_maes),
            "CV_MAE_SD": np.std(fold_maes)
        })

extra_results = pd.DataFrame(rows)

extra_results.to_csv(
    os.path.join(RESULTS, "xgboost_boundary_checks.csv"),
    index=False
)

# Combine original grid and boundary checks
original_grid = pd.read_csv(
    os.path.join(RESULTS, "xgboost_gpu_grid_progress.csv")
)

combined = pd.concat(
    [original_grid, extra_results],
    ignore_index=True
).drop_duplicates(
    subset=[
        "Trait", "learning_rate",
        "n_estimators", "max_depth", "subsample"
    ]
)

best = (
    combined.sort_values(["Trait", "CV_MAE"])
    .groupby("Trait", as_index=False)
    .first()
)

print("\nFINAL XGBOOST PARAMETERS")
print(best.to_string(index=False))

combined.to_csv(
    os.path.join(RESULTS, "xgboost_extended_complete.csv"),
    index=False
)

print("\nBoundary checks completed and saved.")


### Save updated model configurations

Persist the selected configurations from the extended tuning analysis.


In [ ]:

import os
import json
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"
NEW_MODELS = os.path.join(RESULTS, "xgboost_extended_models")
os.makedirs(NEW_MODELS, exist_ok=True)

train_df = pd.read_pickle(
    os.path.join(RESULTS, "train_authors.pkl")
)
test_df = pd.read_pickle(
    os.path.join(RESULTS, "test_authors.pkl")
)

assert len(train_df) == 1094
assert len(test_df) == 274
assert set(train_df["author"]).isdisjoint(
    set(test_df["author"])
)

best_params = {
    "agreeableness": (0.05, 100, 3, 1.0),
    "openness": (0.01, 200, 3, 0.7),
    "conscientiousness": (0.05, 100, 3, 0.7),
    "extraversion": (0.10, 300, 5, 0.7),
    "neuroticism": (0.05, 100, 3, 1.0)
}

rows = []

for trait, (lr, trees, depth, subsample) in best_params.items():
    print(f"Training final XGBoost: {trait}", flush=True)

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=3000,
            min_df=3,
            max_df=0.95,
            ngram_range=(1, 2),
            sublinear_tf=True,
            dtype=np.float32
        )),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            learning_rate=lr,
            n_estimators=trees,
            max_depth=depth,
            subsample=subsample,
            tree_method="hist",
            device="cuda",
            n_jobs=2,
            random_state=42
        ))
    ])

    pipeline.fit(
        train_df["text"],
        train_df[trait]
    )

    pred = pipeline.predict(test_df["text"])
    actual = test_df[trait].to_numpy()

    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)

    rows.append({
        "Trait": trait,
        "Model": "XGBoost Extended",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "learning_rate": lr,
        "n_estimators": trees,
        "max_depth": depth,
        "subsample": subsample
    })

    joblib.dump(
        pipeline,
        os.path.join(NEW_MODELS, f"XGBoost_{trait}.joblib")
    )

    pd.DataFrame(rows).to_csv(
        os.path.join(RESULTS, "xgboost_extended_test.csv"),
        index=False
    )

    print(
        f"{trait}: MAE={mae:.4f}, "
        f"RMSE={rmse:.4f}, R2={r2:.4f}",
        flush=True
    )

print("\nFINAL XGBOOST HELD-OUT TEST RESULTS")
print(pd.DataFrame(rows).round(5).to_string(index=False))

print("\nAll extended XGBoost models and results saved.")


### Check trained-model availability

Confirm which serialized model files are present before subsequent comparisons.


In [ ]:

import os
import joblib

models_dir = (
    "/content/drive/MyDrive/PANDORA_Original/"
    "results/trained_models"
)

for name in [
    "Ridge_agreeableness.joblib",
    "SGD_agreeableness.joblib"
]:
    pipeline = joblib.load(os.path.join(models_dir, name))
    model = pipeline.named_steps["model"]
    tfidf = pipeline.named_steps["tfidf"]

    print("\n", name)
    print("Model parameters:", model.get_params())
    print("TF-IDF parameters:", tfidf.get_params())


### Prepare cross-validation TF–IDF folds

Create the fold-level representations used in the supplementary SGD checks.


In [ ]:

import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

train_df = pd.read_pickle(
    os.path.join(RESULTS, "train_authors.pkl")
)

assert len(train_df) == 1094
assert train_df["author"].nunique() == 1094

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

fold_data = []

cv = GroupKFold(n_splits=5)

for fold_no, (tr, va) in enumerate(
    cv.split(train_df["text"], groups=train_df["author"]),
    start=1
):
    print(f"Preparing SGD fold {fold_no}/5", flush=True)

    tfidf = TfidfVectorizer(
        max_features=10000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    Xtr = tfidf.fit_transform(train_df.iloc[tr]["text"])
    Xva = tfidf.transform(train_df.iloc[va]["text"])

    fold_data.append((tr, va, Xtr, Xva))

print("\nAll five SGD folds are ready.")


### Evaluate additional SGD settings

Compare further regularization choices using the prepared folds.


In [ ]:

import os
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

alphas = [
    1e-9, 1e-7, 1e-6, 1e-5, 1e-4,
    0.0005, 0.001, 0.005, 0.01,
    0.05, 0.1, 0.5, 1.0
]

rows = []

for trait in traits:
    print(f"Checking SGD: {trait}", flush=True)
    y = train_df[trait].to_numpy()

    for alpha in alphas:
        maes = []

        for tr, va, Xtr, Xva in fold_data:
            model = SGDRegressor(
                alpha=alpha,
                loss="squared_error",
                penalty="l2",
                max_iter=3000,
                tol=0.001,
                random_state=42
            )

            model.fit(Xtr, y[tr])
            pred = model.predict(Xva)
            maes.append(mean_absolute_error(y[va], pred))

        rows.append({
            "Trait": trait,
            "Alpha": alpha,
            "CV MAE": np.mean(maes)
        })

grid = pd.DataFrame(rows)

best = (
    grid.sort_values(["Trait", "CV MAE", "Alpha"])
    .groupby("Trait", as_index=False)
    .first()
)

print("\nFINAL SGD RESULTS")
print(best.to_string(index=False))

grid.to_csv(
    os.path.join(RESULTS, "sgd_final_grid.csv"),
    index=False
)

best.to_csv(
    os.path.join(RESULTS, "sgd_final_best.csv"),
    index=False
)

print("\nSGD verification saved.")


### Complete the SGD sensitivity check

Inspect the additional SGD results for the relevant traits.


In [ ]:

import os
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

sgd_alphas = [
    0.0, 1e-12, 1e-11, 1e-10,
    1e-9, 1e-8, 1e-7, 1e-6,
    1e-5, 1e-4
]

rows = []

for trait in ["openness", "neuroticism"]:
    y = train_df[trait].to_numpy()

    for alpha in sgd_alphas:
        scores = []

        for tr, va, Xtr, Xva in fold_data:
            model = SGDRegressor(
                alpha=alpha,
                loss="squared_error",
                penalty="l2",
                max_iter=3000,
                tol=0.001,
                random_state=42
            )

            model.fit(Xtr, y[tr])
            scores.append(
                mean_absolute_error(y[va], model.predict(Xva))
            )

        rows.append({
            "Trait": trait,
            "Alpha": alpha,
            "CV_MAE": np.mean(scores),
            "CV_SD": np.std(scores, ddof=1)
        })

sgd_boundary = pd.DataFrame(rows)

sgd_boundary.to_csv(
    os.path.join(RESULTS, "sgd_boundary_final.csv"),
    index=False
)

for trait in ["openness", "neuroticism"]:
    print(f"\n{trait.upper()}")
    subset = sgd_boundary[
        sgd_boundary["Trait"] == trait
    ].sort_values("Alpha")

    print(subset.to_string(index=False, float_format=lambda x: f"{x:.10g}"))

    best = subset.loc[subset["CV_MAE"].idxmin()]
    print(
        f"BEST: alpha={best['Alpha']:.2e}, "
        f"MAE={best['CV_MAE']:.9f}"
    )


### Compute frozen SBERT author embeddings

Encode author texts with the pretrained sentence-transformer. This section depends on the available runtime and pretrained model.


In [ ]:

import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

train_df = pd.read_pickle(
    os.path.join(RESULTS, "train_authors.pkl")
)

cv = GroupKFold(n_splits=5)
xgb_folds = []

for tr, va in cv.split(
    train_df["text"],
    groups=train_df["author"]
):
    vectorizer = TfidfVectorizer(
        max_features=3000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    Xtr = vectorizer.fit_transform(train_df.iloc[tr]["text"])
    Xva = vectorizer.transform(train_df.iloc[va]["text"])

    xgb_folds.append((tr, va, Xtr, Xva))

# Additional boundary configurations:
# (learning_rate, n_estimators, max_depth, subsample)

configs = {
    "agreeableness": [
        (0.05, 100, 3, 0.9),
        (0.05, 100, 3, 0.5),
    ],
    "conscientiousness": [
        (0.05, 100, 3, 0.9),
        (0.05, 100, 3, 0.6),
    ],
    "extraversion": [
        (0.10, 400, 5, 0.7),
        (0.10, 600, 5, 0.7),
        (0.10, 800, 5, 0.7),
        (0.20, 400, 5, 0.7),
        (0.10, 400, 7, 0.7),
    ],
    "neuroticism": [
        (0.05, 100, 3, 0.9),
        (0.05, 100, 3, 0.5),
    ],
    "openness": [
        (0.002, 200, 3, 0.7),
        (0.005, 300, 3, 0.7),
        (0.01, 400, 3, 0.7),
        (0.01, 600, 3, 0.7),
    ]
}

rows = []
save_path = os.path.join(
    RESULTS, "xgboost_final_boundary_grid.csv"
)

for trait, combinations in configs.items():
    y = train_df[trait].to_numpy()

    for lr, trees, depth, subsample in combinations:
        print(
            f"Testing {trait}: "
            f"lr={lr}, trees={trees}, "
            f"depth={depth}, subsample={subsample}",
            flush=True
        )

        scores = []
        start = time.time()

        for tr, va, Xtr, Xva in xgb_folds:
            model = XGBRegressor(
                objective="reg:squarederror",
                learning_rate=lr,
                n_estimators=trees,
                max_depth=depth,
                subsample=subsample,
                tree_method="hist",
                device="cuda",
                n_jobs=2,
                random_state=42
            )

            model.fit(Xtr, y[tr])
            pred = model.predict(Xva)

            scores.append(
                mean_absolute_error(y[va], pred)
            )

        rows.append({
            "Trait": trait,
            "learning_rate": lr,
            "n_estimators": trees,
            "max_depth": depth,
            "subsample": subsample,
            "CV_MAE": np.mean(scores),
            "CV_MAE_SD": np.std(scores, ddof=1),
            "seconds": time.time() - start
        })

        pd.DataFrame(rows).to_csv(save_path, index=False)

print("\nFINAL BOUNDARY CHECKS")
print(
    pd.DataFrame(rows)
    .sort_values(["Trait", "CV_MAE"])
    .round(7)
    .to_string(index=False)
)


### Use token-aware text chunks

Define chunking that respects the encoder’s token limit.


In [ ]:

import os
import numpy as np

# Precise token-based chunking
tokenizer = model.tokenizer
max_content_tokens = model.max_seq_length - 2

def split_into_token_chunks(text):
    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False,
        truncation=False,
        verbose=False
    )

    if not token_ids:
        return [""]

    return [
        tokenizer.decode(
            token_ids[i:i + max_content_tokens],
            skip_special_tokens=True
        )
        for i in range(0, len(token_ids), max_content_tokens)
    ]

def author_embeddings_corrected(df, split_name):
    save_path = os.path.join(
        RESULTS,
        f"sbert_{split_name}_embeddings_corrected.npy"
    )

    vectors = []

    for i, text in enumerate(df["text"]):
        chunks = split_into_token_chunks(text)

        chunk_vectors = model.encode(
            chunks,
            batch_size=32,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        )

        vectors.append(
            np.mean(chunk_vectors, axis=0)
        )

        if (i + 1) % 100 == 0:
            print(
                f"{split_name}: {i + 1}/{len(df)} authors",
                flush=True
            )

    vectors = np.asarray(vectors, dtype=np.float32)
    np.save(save_path, vectors)

    print("Saved:", save_path)
    return vectors

X_train = author_embeddings_corrected(train_df, "train")
X_test = author_embeddings_corrected(test_df, "test")

print("Training:", X_train.shape)
print("Test:", X_test.shape)


### Fit Ridge on SBERT embeddings

Tune and evaluate a linear regressor on the author-level contextual representations.


In [ ]:

from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import joblib

traits = [
    "agreeableness",
    "openness",
    "conscientiousness",
    "extraversion",
    "neuroticism"
]

alphas = [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 1000]

cv = GroupKFold(n_splits=5)
splits = list(
    cv.split(X_train, groups=train_df["author"])
)

rows = []
grid_rows = []

for trait in traits:
    y_train = train_df[trait].to_numpy()
    y_test = test_df[trait].to_numpy()

    for alpha in alphas:
        fold_maes = []

        for tr, va in splits:
            reg = Ridge(alpha=alpha)
            reg.fit(X_train[tr], y_train[tr])

            pred = reg.predict(X_train[va])
            fold_maes.append(
                mean_absolute_error(y_train[va], pred)
            )

        grid_rows.append({
            "Trait": trait,
            "Alpha": alpha,
            "CV_MAE": np.mean(fold_maes),
            "CV_SD": np.std(fold_maes, ddof=1)
        })

    trait_grid = pd.DataFrame(grid_rows)
    trait_grid = trait_grid[trait_grid["Trait"] == trait]
    best = trait_grid.loc[trait_grid["CV_MAE"].idxmin()]

    reg = Ridge(alpha=float(best["Alpha"]))
    reg.fit(X_train, y_train)
    predictions = reg.predict(X_test)

    rows.append({
        "Trait": trait,
        "Best_Alpha": float(best["Alpha"]),
        "CV_MAE": float(best["CV_MAE"]),
        "CV_SD": float(best["CV_SD"]),
        "Test_MAE": mean_absolute_error(y_test, predictions),
        "Test_RMSE": np.sqrt(
            mean_squared_error(y_test, predictions)
        ),
        "Test_R2": r2_score(y_test, predictions)
    })

    joblib.dump(
        reg,
        os.path.join(RESULTS, f"sbert_ridge_{trait}.joblib")
    )

    print(f"Completed: {trait}", flush=True)

pd.DataFrame(grid_rows).to_csv(
    os.path.join(RESULTS, "sbert_ridge_cv_grid.csv"),
    index=False
)

results = pd.DataFrame(rows)
results.to_csv(
    os.path.join(RESULTS, "sbert_ridge_results.csv"),
    index=False
)

print("\nFINAL SBERT + RIDGE RESULTS")
print(results.round(6).to_string(index=False))


### Check chunk lengths against the tokenizer

Inspect whether the earlier word-based chunks exceed the encoder limit.


In [ ]:

# Check whether 150-word chunks exceed SBERT's token limit

from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

max_tokens = model.max_seq_length

total_chunks = 0
truncated_chunks = 0
max_observed_tokens = 0
token_lengths = []

for df in [train_df, test_df]:
    for author_text in df["text"]:
        chunks = split_into_chunks(author_text)

        for chunk in chunks:
            token_count = len(
                tokenizer.encode(
                    chunk,
                    add_special_tokens=True,
                    truncation=False
                )
            )

            total_chunks += 1
            token_lengths.append(token_count)
            max_observed_tokens = max(
                max_observed_tokens, token_count
            )

            if token_count > max_tokens:
                truncated_chunks += 1

print("SBERT token limit:", max_tokens)
print("Total chunks:", total_chunks)
print("Chunks exceeding limit:", truncated_chunks)
print(
    "Percentage exceeding limit:",
    round(100 * truncated_chunks / total_chunks, 2), "%"
)
print("Maximum observed tokens:", max_observed_tokens)
print("95th percentile:", np.percentile(token_lengths, 95))


### Compare SBERT and TF–IDF Ridge

Evaluate both representations on the same author-disjoint test partition.


In [ ]:

import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

train_df = pd.read_pickle(
    os.path.join(RESULTS, "train_authors.pkl")
)
test_df = pd.read_pickle(
    os.path.join(RESULTS, "test_authors.pkl")
)

assert len(train_df) == 1094
assert len(test_df) == 274
assert set(train_df["author"]).isdisjoint(set(test_df["author"]))

# Best alpha values from the extended Ridge CV search
best_alphas = {
    "agreeableness": 0.5,
    "openness": 1.0,
    "conscientiousness": 1.0,
    "extraversion": 0.1,
    "neuroticism": 0.5
}

rows = []

for trait, alpha in best_alphas.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            min_df=3,
            max_df=0.95,
            ngram_range=(1, 2),
            sublinear_tf=True
        )),
        ("ridge", Ridge(
            alpha=alpha,
            solver="auto",
            tol=0.0001
        ))
    ])

    pipeline.fit(
        train_df["text"].fillna("").astype(str),
        train_df[trait]
    )

    predictions = pipeline.predict(
        test_df["text"].fillna("").astype(str)
    )

    y_true = test_df[trait].to_numpy()

    rows.append({
        "Trait": trait,
        "Alpha": alpha,
        "Test_MAE": mean_absolute_error(
            y_true, predictions
        ),
        "Test_RMSE": np.sqrt(
            mean_squared_error(y_true, predictions)
        ),
        "Test_R2": r2_score(y_true, predictions)
    })

    print(f"Completed: {trait}")

results = pd.DataFrame(rows)

save_path = os.path.join(
    RESULTS,
    "tfidf_ridge_extended_test.csv"
)

results.to_csv(save_path, index=False)

print("\nFINAL TF-IDF + RIDGE TEST RESULTS")
print(results.round(6).to_string(index=False))
print("\nSaved to:", save_path)


### Ablate the XGBoost vocabulary size

Compare the 3,000- and 10,000-feature settings with the documented model configuration.


In [ ]:

import os
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

RESULTS = "/content/drive/MyDrive/PANDORA_Original/results"

train_df = pd.read_pickle(
    os.path.join(RESULTS, "train_authors.pkl")
)
test_df = pd.read_pickle(
    os.path.join(RESULTS, "test_authors.pkl")
)

assert len(train_df) == 1094
assert len(test_df) == 274
assert set(train_df["author"]).isdisjoint(
    set(test_df["author"])
)

# Extended CV-selected XGBoost configurations
params = {
    "agreeableness": (0.05, 100, 3, 1.0),
    "openness": (0.01, 200, 3, 0.7),
    "conscientiousness": (0.05, 100, 3, 0.7),
    "extraversion": (0.10, 600, 5, 0.7),
    "neuroticism": (0.05, 100, 3, 1.0)
}

results = []

for max_features in [3000, 10000]:

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(
        train_df["text"].fillna("").astype(str)
    )
    X_test = vectorizer.transform(
        test_df["text"].fillna("").astype(str)
    )

    for trait, (lr, trees, depth, subsample) in params.items():

        model = XGBRegressor(
            learning_rate=lr,
            n_estimators=trees,
            max_depth=depth,
            subsample=subsample,
            objective="reg:squarederror",
            tree_method="hist",
            device="cuda",
            n_jobs=2,
            random_state=42
        )

        model.fit(X_train, train_df[trait])

        pred = model.predict(X_test)

        results.append({
            "Trait": trait,
            "Features": max_features,
            "Test_MAE": mean_absolute_error(
                test_df[trait], pred
            )
        })

        print(
            f"Completed: {trait}, {max_features} features",
            flush=True
        )

results_df = pd.DataFrame(results)

results_df.to_csv(
    os.path.join(
        RESULTS,
        "xgboost_vocabulary_ablation.csv"
    ),
    index=False
)

print("\nXGBOOST VOCABULARY ABLATION")
print(
    results_df.pivot(
        index="Trait",
        columns="Features",
        values="Test_MAE"
    ).round(6).to_string()
)


### Reconnect Google Drive for the final ablation

This mount was repeated when the Colab runtime was restarted; it is retained as executed.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


### Locate the original data files

Check the folder contents before loading raw comments again.


In [ ]:

import os

base = "/content/drive/MyDrive/PANDORA_Original"

print("Folder postoji:", os.path.exists(base))

for root, dirs, files in os.walk(base):
    for filename in files:
        if filename.endswith((".csv", ".pkl", ".parquet", ".jsonl")):
            print(os.path.join(root, filename))


### Verify the raw CSV schemas

Confirm the author and comment-body columns used in the text-quantity ablation.


In [ ]:

import pandas as pd
import os

BASE = "/content/drive/MyDrive/PANDORA_Original"

for filename in [
    "author_profiles.csv",
    "pandora_filtered_comments.csv"
]:
    path = os.path.join(BASE, filename)
    df = pd.read_csv(path, nrows=3)

    print(f"\n{filename}")
    print("Kolone:", df.columns.tolist())
    print(df.head(2).to_string(index=False, max_colwidth=60))

train = pd.read_pickle(
    os.path.join(BASE, "results/train_authors.pkl")
)

print("\ntrain_authors.pkl kolone:")
print(train.columns.tolist())


### Ablate comments per author

Compare 50, 100, and 500 nested sampled comments per author using fixed Ridge settings and the same held-out split.


In [ ]:

import os
import gc
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = "/content/drive/MyDrive/PANDORA_Original"
RESULTS = os.path.join(BASE, "results")

train = pd.read_pickle(os.path.join(RESULTS, "train_authors.pkl"))
test = pd.read_pickle(os.path.join(RESULTS, "test_authors.pkl"))

traits = [
    "agreeableness", "openness", "conscientiousness",
    "extraversion", "neuroticism"
]

alphas = {
    "agreeableness": 0.5,
    "openness": 1.0,
    "conscientiousness": 1.0,
    "extraversion": 0.1,
    "neuroticism": 0.5
}

# Preserve the original author-disjoint split
train_authors = set(train["author"].astype(str))
test_authors = set(test["author"].astype(str))
assert train_authors.isdisjoint(test_authors)

all_authors = train_authors | test_authors

# Load original individual comments
comments = pd.read_csv(
    os.path.join(BASE, "pandora_filtered_comments.csv"),
    usecols=["author", "body"],
    dtype={"author": str, "body": str}
)

comments = comments[comments["author"].isin(all_authors)].copy()
comments = comments.dropna(subset=["body"])
comments["body"] = comments["body"].str.strip()
comments = comments[comments["body"].str.len() > 0]

print("Valid comments:", len(comments))
print("Authors:", comments["author"].nunique())

# One deterministic random order per author, shared across all caps.
# Thus, the 50-comment sample is nested inside 100 and 500.
comments = comments.sample(frac=1, random_state=42).copy()
comments["comment_rank"] = comments.groupby("author").cumcount()

rows = []

for cap in [50, 100, 500]:
    print(f"\nProcessing cap = {cap}", flush=True)

    selected = comments[comments["comment_rank"] < cap]

    aggregated = (
        selected.groupby("author", sort=False)["body"]
        .agg(" ".join)
    )

    train_text = (
        train["author"].astype(str).map(aggregated).fillna("")
    )
    test_text = (
        test["author"].astype(str).map(aggregated).fillna("")
    )

    assert (train_text.str.len() > 0).all()
    assert (test_text.str.len() > 0).all()

    vectorizer = TfidfVectorizer(
        max_features=10000,
        min_df=3,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(train_text)
    X_test = vectorizer.transform(test_text)

    for trait in traits:
        model = Ridge(
            alpha=alphas[trait],
            solver="auto",
            tol=0.0001
        )

        model.fit(X_train, train[trait])
        predictions = model.predict(X_test)

        rows.append({
            "Trait": trait,
            "Comment_Cap": cap,
            "Test_MAE": mean_absolute_error(
                test[trait], predictions
            ),
            "Test_RMSE": np.sqrt(mean_squared_error(
                test[trait], predictions
            )),
            "Test_R2": r2_score(
                test[trait], predictions
            )
        })

        print(f"  Completed: {trait}", flush=True)

    del X_train, X_test, vectorizer
    gc.collect()

results = pd.DataFrame(rows)

output_path = os.path.join(
    RESULTS, "comment_cap_ablation.csv"
)
results.to_csv(output_path, index=False)

print("\nFINAL COMMENT CAP ABLATION — TEST MAE")
print(
    results.pivot(
        index="Trait",
        columns="Comment_Cap",
        values="Test_MAE"
    ).round(6).to_string()
)

print("\nSaved to:", output_path)


### Inspect the final Colab environment

Print the versions available at the end of the notebook. These are not necessarily the versions used in every earlier run.


In [ ]:

import sys
import sklearn
import xgboost
import numpy
import pandas
import scipy

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
